Wikidata Preparation Notebook
Project: Distractor Filtering Through Open KG Concept Proximity for Dense RAG

This notebook covers:
1. Wikipedia corpus exploration and passage segmentation
2. Entity linking with ReFiNed
3. Wikidata KG subgraph extraction via SPARQL

Pipeline reference: Steps 1-3 of the proposal


## Step 0 — Configuration & Corpus Loading

**Corpus**: `psgs_w100.tsv` — the DPR (Dense Passage Retrieval) standard Wikipedia corpus.

Karpukhin et al. (2020) created this corpus for their DPR paper:
- Took the **Wikipedia dump of Dec 20, 2018**
- Cleaned it (removed tables, infoboxes, disambiguation pages)
- Split every article into **non-overlapping passages of exactly 100 words** (mechanical cut, no sentence alignment)
- Result: **~21M passages**, each with an `id`, `text`, and `title`

This file (`psgs_w100.tsv`) is the **de facto standard** for open-domain QA benchmarks.
Nearly all subsequent papers (Contriever, FiD, ATLAS, RAG, Silvestri) use this exact corpus to ensure result comparability.

Why Dec 2018? Because **NQ-open (Natural Questions)** was annotated on that Wikipedia version — the gold answers are extracted from there. Using a different version would cause misalignment.

In [1]:
from pathlib import Path

# --- Configuration ---
PROJECT_ROOT = Path.cwd()

# DPR corpus: psgs_w100.tsv (Karpukhin et al., 2020)
# ~21M passages of 100 words from Wikipedia Dec 2018.
# This is the standard benchmark corpus for NQ-open and most open-domain QA papers.
CORPUS_PATH = PROJECT_ROOT / "data" / "wikipedia_2018" / "psgs_w100.tsv"

print(f"Corpus file: {CORPUS_PATH}")
print(f"Exists: {CORPUS_PATH.exists()}")

if CORPUS_PATH.exists():
    size_gb = CORPUS_PATH.stat().st_size / (1024 ** 3)
    print(f"Size: {size_gb:.1f} GB")

Corpus file: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018\psgs_w100.tsv
Exists: True
Size: 12.8 GB


## Step 1 — Corpus Exploration
Understand the structure of the DPR Wikipedia corpus: columns, dtypes, row counts, and text format.
The file is a TSV with columns `id`, `text`, `title` (~21M rows, ~13 GB).
We load a sample first to avoid memory issues.

In [2]:
import pandas as pd

# Load only first 10 rows to inspect schema without loading 13 GB into RAM.
# The TSV uses tab separator; text fields are double-quoted.
sample_df = pd.read_csv(CORPUS_PATH, sep="\t", nrows=10)

print(f"Columns: {list(sample_df.columns)}")
print(f"Dtypes:\n{sample_df.dtypes}\n")
sample_df.head(5)

Columns: ['id', 'text', 'title']
Dtypes:
id       int64
text       str
title      str
dtype: object



,id,text,title
0,1,"Aaron Aaron ( or ; ""Ahärôn"") is a prophet, hig...",Aaron
1,2,God at Sinai granted Aaron the priesthood for ...,Aaron
2,3,his rod turn into a snake. Then he stretched o...,Aaron
3,4,"however, Aaron and Hur remained below to look ...",Aaron
4,5,"Aaron and his sons to the priesthood, and arra...",Aaron


In [3]:
# Inspect a single passage to see the actual text content and word count
passage = sample_df.iloc[0]
word_count = len(passage["text"].split())

print(f"ID:    {passage['id']}")
print(f"Title: {passage['title']}")
print(f"Words: {word_count}")
print(f"Text:\n{passage['text']}")

ID:    1
Title: Aaron
Words: 100
Text:
Aaron Aaron ( or ; "Ahärôn") is a prophet, high priest, and the brother of Moses in the Abrahamic religions. Knowledge of Aaron, along with his brother Moses, comes exclusively from religious texts, such as the Bible and Quran. The Hebrew Bible relates that, unlike Moses, who grew up in the Egyptian royal court, Aaron and his elder sister Miriam remained with their kinsmen in the eastern border-land of Egypt (Goshen). When Moses first confronted the Egyptian king about the Israelites, Aaron served as his brother's spokesman ("prophet") to the Pharaoh. Part of the Law (Torah) that Moses received from


## Step 1b — Towards sentence-aligned segmentation

### The problem: semantic bleeding
The DPR corpus cuts articles every 100 words **mechanically**, ignoring sentence boundaries.
A sentence can start in passage N and end in passage N+1. This causes:

1. **Entity linking degradation**: ReFiNed may fail to link entities in truncated sentences
2. **LLM reader confusion**: incomplete sentences at chunk boundaries add noise to Llama2's context

### Plan
1. **Reconstruct full articles**: group DPR passages by `title` and concatenate `text` (ordered by `id`) to recover the original article text
2. **Re-segment with sentence alignment**: apply a sentence-aware algorithm (details TBD — will review the approach from `base/preprocessing.ipynb` first)
3. **Save as `psgs_w100_sentence.tsv`**: same TSV format (`id`, `text`, `title`), new passage boundaries

### Dual-corpus strategy
We keep **both** versions:
- `psgs_w100.tsv` — DPR standard (baseline, results comparable to literature)
- `psgs_w100_sentence.tsv` — sentence-aligned variant (experimental)

This enables an **ablation study**: how much improvement comes from better segmentation
vs. from KG reranking? Without both corpora, this question is unanswerable.

### Step 1b.1 — Reconstruct full articles from DPR passages
Group passages by `title`, concatenate `text` ordered by `id`.
The file is ~13 GB so we process it in chunks to avoid memory issues.

In [4]:
import polars as pl

# --- Article reconstruction from DPR passages ---
# The DPR corpus splits articles into 100-word chunks without sentence alignment.
# To re-segment with sentence boundaries, we first need to reconstruct full articles.
#
# Strategy:
#   1. scan_csv       -> lazy read (does NOT load 13 GB into RAM)
#   2. group_by       -> partition passages by article title (parallel across all cores)
#   3. sort_by("id")  -> order passages WITHIN each group (not globally — much faster)
#   4. str.concat     -> join passage texts into a single article string
#   5. collect(engine="streaming") -> execute in streaming mode (processes data in batches)
#
# Why sort_by inside agg instead of global sort:
#   Global sort on 21M rows is a blocking operation that kills parallelism.
#   Sorting within each group (typically 1-50 passages per article) is trivial
#   and runs in parallel across groups.

articles_df = (
    pl.scan_csv(CORPUS_PATH, separator="\t")
    .group_by("title")
    .agg(
        # Sort passages by id WITHIN each article, then concatenate
        pl.col("text").sort_by("id").str.concat(" ").alias("full_text"),
        # Keep track of how many DPR passages made up this article
        pl.col("id").count().alias("n_passages"),
    )
    .collect(engine="streaming")
)
articles_df.head(2)

C:\Users\Utente\AppData\Local\Temp\ipykernel_11096\3808458081.py:24: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  pl.col("text").sort_by("id").str.concat(" ").alias("full_text"),


title,full_text,n_passages
str,str,u32
"""Rural Municipality of Chester …","""Rural Municipality of Chester …",2
"""Bell Bajao""","""Bell Bajao Bell Bajao (Hindi f…",6


In [5]:
articles_df.filter(pl.col("title") == "PAEEK")

title,full_text,n_passages
str,str,u32
"""PAEEK""","""PAEEK PAEEK (; ""Podosfairiki A…",4


In [6]:
original_df = (
    pl.scan_csv(CORPUS_PATH, separator="\t")
    .collect(engine="streaming")
)

In [7]:
original_text = ""
for row in original_df.filter(pl.col("title") == "PAEEK").get_column("text"):
    original_text += row

In [8]:
original_text

'PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias" = "Football and sport union of Kyrenia District") is a Cypriot sports club founded in Kyrenia in 1953 by graduates of Kyrenia Gymnasium and represented the first force to be reckoned from the small city. It now plays in exile in Nicosia since the Turkish invasion of Cyprus in July 1974. This union has football division competing in Cypriot Second Division. PAEEK used to have also a basketball division but due to economic difficulties had to suspend it for some years up to date. The PAEEK was a founding member of theCyprus Basketball Federation in 1966 rising to fame in the early 1970s after its basketball division won the Cyprus top division basketball league in 3 consecutive years.. The PAEEK reached the Cyprus Basketball Cup final on 5 occasions losing them all . 1995 was APOEL\'s year, when they took the basketball double. As losers of the Cup final, PAEEK automatically qualified to represent Cyprus in Europe in the S

In [9]:
import re

# --- Padding detection: build prefix from start, check if it's at the end ---
# The DPR padding copies the first K words of the article to fill the last chunk.
# So the text ENDS with a copy of its own beginning.
#
# Algorithm:
#   1. Take the first n words → head_str
#   2. Check if the text ends with head_str (regex: head_str$)
#   3. Keep incrementing n — the maximum match is the full padding

paeek_text = articles_df.filter(pl.col("title") == "PAEEK")["full_text"][0]
words = paeek_text.split(" ")
print(f"Total words: {len(words)}")
print(f"First 8: {' '.join(words[:8])}")
print(f"Last 8:  {' '.join(words[-8:])}")
print()

padding_len = 0
for n in range(1, 101):
    head_str = " ".join(words[:n])
    # endswith head_str, preceded by a space (word boundary)
    if paeek_text.endswith(" " + head_str):
        padding_len = n

if padding_len > 0:
    padding_str = " ".join(words[:padding_len])
    # Remove trailing padding (+ the space before it)
    clean_text = paeek_text[:-(len(padding_str) + 1)]
    clean_words = clean_text.split(" ")
    print(f"Padding found: {padding_len} words")
    print(f"Clean article: {len(clean_words)} words")
    print(f"Last 10 clean words: ...{' '.join(clean_words[-10:])}")
    print(f"Padding (first 10 words): {' '.join(words[:min(10, padding_len)])}...")
else:
    print("No padding detected")

Total words: 400
First 8: PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias"
Last 8:  Nicosia since the Turkish invasion of Cyprus in

Padding found: 57 words
Clean article: 343 words
Last 10 clean words: ...yellow and black as they were before the Turkish invasion.
Padding (first 10 words): PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias" = "Football...


### Step 1b.2 — Strip DPR padding and save clean articles

The padding detection works (tested on PAEEK: 57 words of padding found, article
ends with a coherent sentence).

Now we:
1. Define `strip_dpr_padding()` as a reusable function
2. Apply it to all ~5.9M articles in `articles_df`
3. Save the clean articles as TSV in `data/wikipedia_2018_clean/`

In [8]:
def strip_dpr_padding(text: str) -> tuple[str, int, int]:
    """Remove DPR trailing padding from a reconstructed article.

    The DPR corpus pads the last chunk of each article by copying words
    from the beginning of the article to fill the last passage to exactly
    100 words. This function detects the longest prefix of the text that
    also appears as a suffix, and strips it.

    Also computes word counts so the caller doesn't need to re-split
    the text (which is expensive at scale on 3.2M articles).

    Args:
        text: full article text (concatenation of all DPR passages).

    Returns:
        Tuple of (clean_text, original_word_count, clean_word_count).
    """
    words = text.split(" ")
    original_words = len(words)

    # Single-passage articles (<=100 words) cannot have padding:
    # the entire text IS the article content.
    if original_words <= 100:
        return text, original_words, original_words

    # The padding is at most 99 words (one 100-word chunk minus at least 1
    # original word). We check each possible prefix length and keep the
    # maximum that matches the suffix — that's the true padding length.
    padding_len = 0
    for n in range(1, 100):
        head_str = " ".join(words[:n])
        # Prepend space to enforce word boundary:
        # without it, prefix "art" could match suffix "...restart"
        if text.endswith(" " + head_str):
            padding_len = n

    if padding_len == 0:
        return text, original_words, original_words

    # Slice off the padding and the space before it
    padding_str = " ".join(words[:padding_len])
    clean_text = text[:-(len(padding_str) + 1)]
    clean_words = original_words - padding_len

    return clean_text, original_words, clean_words


# --- Quick verification on PAEEK ---
paeek_text = articles_df.filter(pl.col("title") == "PAEEK")["full_text"][0]
clean, orig_w, clean_w = strip_dpr_padding(paeek_text)
print(f"PAEEK: {orig_w} words → {clean_w} clean (padding: {orig_w - clean_w})")
print(f"Ends with: ...{' '.join(clean.split(' ')[-8:])}")

NameError: name 'articles_df' is not defined

In [11]:
import sys
import time
import gc
from multiprocessing import Pool, cpu_count

# === Padding removal via multiprocessing ===
#
# strip_dpr_padding is a pure Python function (string split + endswith checks).
# We parallelize it using multiprocessing.Pool, which creates separate OS processes.
#
# --- Why not threads? The GIL (Global Interpreter Lock) ---
# CPython has a mutex called the GIL that allows only ONE thread to execute
# Python bytecode at a time. Even with 16 threads, only 1 runs while the
# others wait. This makes threading useless for CPU-bound work like ours.
# Processes sidestep this: each process is a full Python interpreter with
# its OWN GIL, so N processes = N cores truly busy in parallel.
#
# --- Why the function lives in utils/text_processing.py ---
# On Windows, multiprocessing uses "spawn" to create workers: each worker
# starts a brand-new Python interpreter and must IMPORT the target function
# from a .py module file. Functions defined only in notebook cells cannot be
# imported this way (they exist only in the kernel's memory). The previous
# cell writes the function to utils/text_processing.py via inspect.getsource,
# keeping the notebook as the single source of truth.
#
# --- Pool.imap(fn, data, chunksize) vs Pool.map ---
# Both distribute work across workers. The difference:
#   - map:  collects ALL results into a list, returns when everything is done.
#           With 3.2M results, this list alone uses ~1-2 GB of RAM.
#   - imap: returns an ITERATOR that yields results one at a time as workers
#           finish. No intermediate list of 3.2M tuples — we unpack each tuple
#           immediately and append to the three target lists.
#
# --- IPC (Inter-Process Communication) ---
# Workers and the main process exchange data via OS pipes. Python objects are
# serialized with pickle, sent as bytes through the pipe, and deserialized on
# the other side. chunksize=1000 means each IPC round-trip carries 1000 articles,
# reducing overhead to ~3200 calls instead of 3.2M.
#
# --- Return value ---
# Each worker returns a tuple (clean_text, original_words, clean_words).
# Word counts are computed inside the worker while the text is already split,
# avoiding a separate str.split pass on 3.2M articles in Polars afterwards.

# Free memory before starting: original_df (21M DPR passages) is no longer needed
if "original_df" in dir():
    del original_df
    gc.collect()

# Ensure project root is in Python's import path so "from utils..." works
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the function from the .py file (auto-generated by the previous cell).
# We use the alias _worker_fn because:
#   - multiprocessing workers on Windows can only call functions importable from
#     a .py module, NOT functions defined in notebook cells
#   - the alias avoids overwriting the notebook-defined strip_dpr_padding
#   - the .py file is kept in sync by inspect.getsource in the previous cell
import importlib
import utils.text_processing
importlib.reload(utils.text_processing)
from utils.text_processing import strip_dpr_padding as _worker_fn

# .to_list() extracts the Polars Series as a plain Python list of strings.
# From here on we work in pure Python — no Polars intermediate structures.
texts = articles_df["full_text"].to_list()
n_cores = max(1, cpu_count() // 2)

print(f"Processing {len(texts):,} articles on {n_cores} cores (chunksize=1000)")

# --- imap: iterate results as they arrive ---
# Each result is a tuple (clean_text, original_words, clean_words).
# We unpack and append incrementally — no giant intermediate results list,
# no zip(*results) transpose operation.
clean_texts = []
original_words = []
clean_words = []

start = time.time()

with Pool(n_cores) as pool:
    for text, orig_w, clean_w in pool.imap(_worker_fn, texts, chunksize=1000):
        clean_texts.append(text)
        original_words.append(orig_w)
        clean_words.append(clean_w)

elapsed = time.time() - start
print(f"Pool done in {elapsed:.1f}s")

del texts
gc.collect()

# --- Put results back into the DataFrame ---
# pl.Series("name", python_list) creates a Polars Series from a Python list.
# with_columns adds/replaces columns in the existing DataFrame.
# No Polars str.split needed — word counts were computed inside the workers.
articles_df = articles_df.with_columns(
    pl.Series("clean_text", clean_texts),
    pl.Series("original_words", original_words, dtype=pl.UInt32),
    pl.Series("clean_words", clean_words, dtype=pl.UInt32),
)

del clean_texts, original_words, clean_words
gc.collect()

# --- Verification: PAEEK (known: 400 words → 343 clean, padding = 57) ---
paeek = articles_df.filter(pl.col("title") == "PAEEK")
paeek_orig = paeek["original_words"][0]
paeek_clean = paeek["clean_words"][0]
paeek_padding = paeek_orig - paeek_clean
print(f"\nVerification — PAEEK:")
print(f"  Original: {paeek_orig} words → Clean: {paeek_clean} words")
print(f"  Padding removed: {paeek_padding} (expected: 57)")
assert paeek_padding == 57, f"PAEEK mismatch! Got {paeek_padding}"
print("  ✓ Check passed")

# --- Summary statistics ---
total_orig = articles_df["original_words"].sum()
total_clean = articles_df["clean_words"].sum()
total_padding = total_orig - total_clean
n_with_padding = articles_df.filter(
    pl.col("original_words") != pl.col("clean_words")
).shape[0]

print(f"\nSummary:")
print(f"  Articles with padding removed: {n_with_padding:,} / {len(articles_df):,}")
print(f"  Total words before: {total_orig:,}")
print(f"  Total words after:  {total_clean:,}")
print(f"  Padding removed:    {total_padding:,} ({total_padding/total_orig*100:.1f}%)")

Processing 3,232,908 articles on 12 cores (chunksize=1000)
Pool done in 103.5s

Verification — PAEEK:
  Original: 400 words → Clean: 343 words
  Padding removed: 57 (expected: 57)
  ✓ Check passed

Summary:
  Articles with padding removed: 3,193,309 / 3,232,908
  Total words before: 2,101,532,400
  Total words after:  1,931,717,612
  Padding removed:    169,814,788 (8.1%)


In [12]:
# --- Save clean articles as TSV ---
# This is an intermediate artifact: one row per article, with the full clean text.
# It will be the input for sentence-aligned re-segmentation (next step).

CLEAN_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PATH = CLEAN_DIR / "articles_clean.tsv"

# Select and rename columns for the output file:
#   - title:                Wikipedia article title
#   - text:                 cleaned full article (padding removed)
#   - word_count:           number of words in clean text (for quick filtering later)
#   - original_n_passages:  how many DPR chunks made up this article (for traceability)
output_df = articles_df.select(
    pl.col("title"),
    pl.col("clean_text").alias("text"),
    pl.col("clean_words").alias("word_count"),
    pl.col("n_passages").alias("original_n_passages"),
)

# write_csv with separator="\t" produces a TSV file.
# Polars handles quoting of fields that contain tabs or newlines automatically.
output_df.write_csv(CLEAN_PATH, separator="\t")

size_gb = CLEAN_PATH.stat().st_size / (1024 ** 3)
print(f"Saved to: {CLEAN_PATH}")
print(f"Size: {size_gb:.1f} GB")
print(f"Rows: {len(output_df):,}")
print(f"\nSample:")
output_df.head(3)

Saved to: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_clean\articles_clean.tsv
Size: 11.2 GB
Rows: 3,232,908

Sample:


title,text,word_count,original_n_passages
str,str,u32,u32
"""Rural Municipality of Chester …","""Rural Municipality of Chester …",200,2
"""Bell Bajao""","""Bell Bajao Bell Bajao (Hindi f…",520,6
"""Myanmar–Thailand relations""","""Myanmar–Thailand relations Bur…",1231,13


In [2]:
import polars as pl
from pathlib import Path

# --- Resume guard: load clean articles from disk if not already in memory ---
# When resuming in a new session, previous cells haven't been executed,
# so output_df doesn't exist. We load the saved TSV instead of re-running
# the entire corpus reconstruction + padding removal pipeline (~2 min).

if "output_df" not in dir():
    PROJECT_ROOT = Path.cwd()
    CLEAN_PATH = PROJECT_ROOT / "data" / "wikipedia_2018_clean" / "articles_clean.tsv"

    print(f"output_df not found in memory — loading from disk...")
    print(f"Source: {CLEAN_PATH}")

    output_df = pl.read_csv(CLEAN_PATH, separator="\t")

    print(f"Loaded: {len(output_df):,} articles")
    print(f"Columns: {output_df.columns}")
    print(f"Schema: {dict(output_df.schema)}")
    output_df.head(3)
else:
    print(f"output_df already in memory ({len(output_df):,} rows) — skipping load")

output_df not found in memory — loading from disk...
Source: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_clean\articles_clean.tsv
Loaded: 3,232,908 articles
Columns: ['title', 'text', 'word_count', 'original_n_passages']
Schema: {'title': String, 'text': String, 'word_count': Int64, 'original_n_passages': Int64}


### Step 1b.3 — Sentence-aligned re-segmentation

**Goal**: re-segment each article into passages of exactly 100 words with complete sentences.

**Algorithm (per article)**:
1. Split into sentences using regex (`(?<=[.!?])\s+`)
2. Greedy accumulation: add sentences to the current segment while `word_count < 100`
3. When the next sentence would push the count to ≥ 100, close the segment and start a new one
4. Sentences > 100 words: mechanical split at 100-word boundaries; if the final fragment is < 100 words, take the last 100 words of the sentence (backward slide, overlapping with the previous chunk)
5. Normal segments < 100 words: pad to 100 using words from the article's first segment

**Why pad from the first segment**: mirrors DPR's own strategy — gives every passage an explicit anchor to the article's identity (title + opening lines).

**Why backward slide instead of padding for long sentences**: preserves contextual contiguity. The overlap with the previous chunk is the accepted trade-off (sentences > 100 words are extremely rare in Wikipedia).

**Regex limitations**: false positives on abbreviations ("U.S. Army" → splits after "U.S."), but harmless — short fragments simply get merged into the same segment by the accumulator.

In [9]:
import re

# Pre-compiled regex for sentence splitting.
# Compiled once here in the notebook; in the .py module it's compiled
# once per worker at import time. Either way, avoids re-compilation
# on every segment_article call.
_SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\s+')


def segment_article(text: str) -> list[str]:
    """Re-segment article text into 100-word passages with sentence alignment.

    Algorithm:
      1. Split text into sentences (regex on .!? followed by whitespace)
      2. Greedily pack sentences into segments while word_count < 100
      3. Sentences > 100 words: mechanical split at 100-word boundaries;
         final fragment < 100 words uses backward slide (last 100 words
         of the sentence) to preserve contextual contiguity
      4. Normal segments < 100 words: pad to exactly 100 words with words
         from the article's first segment (mirrors DPR padding strategy)

    Args:
        text: Clean article text (DPR padding already removed).

    Returns:
        List of passage strings, each exactly 100 words.
    """
    # Sentence split using pre-compiled regex.
    # Imperfect for abbreviations ("U.S. Army" splits after "U.S.") but
    # harmless: short fragments get merged into the same segment.
    sentences = _SENTENCE_SPLIT.split(text)

    # --- Greedy sentence accumulation ---
    # Pack sentences into segments, keeping word count strictly < 100.
    # When the next sentence would push the count to >= 100, close the
    # current segment and start a new one with that sentence.
    segments = []
    current = []
    current_wc = 0

    for sent in sentences:
        # count(' ') + 1: word count without allocating a list.
        # Equivalent to len(split(' ')) but O(1) memory (no list created).
        sent_wc = sent.count(' ') + 1

        if sent_wc >= 100:
            # --- Long sentence handler ---
            # Close any accumulated segment first
            if current:
                segments.append(' '.join(current))
                current = []
                current_wc = 0

            # Mechanical split at 100-word boundaries
            words = sent.split(' ')
            while len(words) >= 100:
                segments.append(' '.join(words[:100]))
                words = words[100:]

            # Final fragment < 100 words: backward slide.
            # Take the last 100 words of the original sentence to preserve
            # contextual contiguity (accepts overlap with previous chunk).
            if words:
                all_words = sent.split(' ')
                segments.append(' '.join(all_words[-100:]))
        elif current and current_wc + sent_wc >= 100:
            # Adding this sentence would reach/exceed 100 — close segment
            segments.append(' '.join(current))
            current = [sent]
            current_wc = sent_wc
        else:
            current.append(sent)
            current_wc += sent_wc

    # Flush remaining sentences as the last segment
    if current:
        segments.append(' '.join(current))

    if not segments:
        return []

    # --- Pad to exactly 100 words ---
    # Source: words from the first segment (same strategy as DPR, which pads
    # the last chunk from the article's beginning). Gives every passage an
    # explicit anchor to the article's identity.
    #
    # Circular repetition: if first_words has fewer words than padding_needed,
    # we cycle through them repeatedly. This guarantees exactly 100 words
    # even for very short articles (e.g., 30-word article needs 70 padding
    # words — cycles through the 30 words ~2.3 times).
    first_words = segments[0].split(' ')
    n_first = len(first_words)
    padded = []

    for seg in segments:
        n = seg.count(' ') + 1
        if n < 100:
            padding_needed = 100 - n
            # Modular indexing: word at position i comes from first_words[i % n_first]
            padding_words = [first_words[i % n_first] for i in range(padding_needed)]
            padded.append(seg + ' ' + ' '.join(padding_words))
        else:
            padded.append(seg)

    return padded


# --- Test on PAEEK (known: 343 clean words, 4 DPR passages) ---
paeek_text = output_df.filter(pl.col("title") == "PAEEK")["text"][0]
print(f"PAEEK clean text: {paeek_text.count(' ') + 1} words\n")

segments = segment_article(paeek_text)
print(f"Segments: {len(segments)}\n")

for i, seg in enumerate(segments):
    wc = seg.count(' ') + 1
    words = seg.split(' ')
    start = ' '.join(words[:8])
    end = ' '.join(words[-8:])
    print(f"  [{i}] {wc} words")
    print(f"      start: {start} ...")
    print(f"      end:   ... {end}")
    print()

# --- Test circular padding on a short article ---
short_test = "This is a very short article with only twelve words in it."
short_segs = segment_article(short_test)
short_wc = short_segs[0].count(' ') + 1
print(f"Short article test ({short_test.count(' ') + 1} words):")
print(f"  Padded to: {short_wc} words (expected: 100)")
assert short_wc == 100, f"Circular padding failed! Got {short_wc}"
print(f"  ✓ Circular padding works")

PAEEK clean text: 343 words

Segments: 4

  [0] 100 words
      start: PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias" ...
      end:   ... PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias"

  [1] 100 words
      start: The PAEEK was a founding member of the ...
      end:   ... of Kyrenia District") is a Cypriot sports club

  [2] 100 words
      start: They were knocked out by PAOK BC Salonika ...
      end:   ... "Podosfairiki Athlitiki Enosi Eparxeias Kerynias" = "Football and

  [3] 100 words
      start: In the 2011–2012 season they came closest to ...
      end:   ... "Football and sport union of Kyrenia District") is

Short article test (12 words):
  Padded to: 100 words (expected: 100)
  ✓ Circular padding works


In [10]:
from pathlib import Path
import polars as pl

# --- File-based multiprocessing worker functions ---
# These functions are used by Pool workers for the file-based shared-nothing
# parallelization strategy. They read/write TSV files independently —
# only an integer (fragment index) travels through IPC.

# Worker-local globals, set by _init_file_worker via Pool initializer.
# Each spawned process calls the initializer once at startup, storing the
# paths in its own address space. The actual worker function then reads
# these globals — no need to pickle paths on every call.
_worker_input_dir: Path | None = None
_worker_output_dir: Path | None = None


def _init_file_worker(input_dir: str, output_dir: str) -> None:
    """Pool initializer: store I/O paths in worker-local globals."""
    global _worker_input_dir, _worker_output_dir
    _worker_input_dir = Path(input_dir)
    _worker_output_dir = Path(output_dir)


def file_segment_worker(frag_idx: int) -> int:
    """Process one input fragment, write output fragment.

    Reads input_dir/frag_{idx}.tsv (columns: title, text),
    applies segment_article to each article, writes flat passages
    to output_dir/frag_{idx}.tsv (columns: text, title).

    Resumability: if output file already exists, skip processing.

    Args:
        frag_idx: fragment index (0-99).

    Returns:
        Number of passages produced (-1 if skipped due to resumability).
    """
    input_path = _worker_input_dir / f"frag_{frag_idx}.tsv"
    output_path = _worker_output_dir / f"frag_{frag_idx}.tsv"

    # Resumability: skip if output already exists
    if output_path.exists():
        return -1

    df = pl.read_csv(input_path, separator="\t")

    titles = df["title"].to_list()
    texts = df["text"].to_list()

    # Build flat passage rows: one (text, title) per passage
    out_texts = []
    out_titles = []

    for title, text in zip(titles, texts):
        passages = segment_article(text)
        for passage in passages:
            out_texts.append(passage)
            out_titles.append(title)

    result_df = pl.DataFrame({
        "text": out_texts,
        "title": out_titles,
    })
    result_df.write_csv(output_path, separator="\t")

    return len(out_texts)


print("Worker functions defined: _init_file_worker, file_segment_worker")

Worker functions defined: _init_file_worker, file_segment_worker


In [11]:
import inspect

# === Write all functions to utils/text_processing.py ===
#
# Single source of truth: functions are DEFINED in notebook cells above,
# then written to the .py module here via inspect.getsource.
# On Windows, multiprocessing uses "spawn" — workers must IMPORT functions
# from a .py file (notebook-defined functions can't be pickled for import).
#
# This cell should be re-run whenever any function definition changes.

utils_dir = PROJECT_ROOT / "utils"
utils_dir.mkdir(exist_ok=True)
(utils_dir / "__init__.py").touch()

# --- Assemble module content ---
# Header: imports and module-level constants that functions reference.
# Functions: extracted verbatim via inspect.getsource.
MODULE_HEADER = '''\
import re
from pathlib import Path

import polars as pl


# Pre-compiled regex for sentence splitting (compiled once per worker at import).
_SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\\s+')


# Worker-local globals, set by _init_file_worker via Pool initializer.
_worker_input_dir: Path | None = None
_worker_output_dir: Path | None = None

'''

module_content = MODULE_HEADER
module_content += inspect.getsource(strip_dpr_padding) + "\n\n"
module_content += inspect.getsource(segment_article) + "\n\n"
module_content += inspect.getsource(_init_file_worker) + "\n\n"
module_content += inspect.getsource(file_segment_worker) + "\n"

module_path = utils_dir / "text_processing.py"
module_path.write_text(module_content, encoding="utf-8")

print(f"Written to: {module_path}")
print(f"Functions: strip_dpr_padding, segment_article, _init_file_worker, file_segment_worker")
print(f"Size: {module_path.stat().st_size:,} bytes")

Written to: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\utils\text_processing.py
Functions: strip_dpr_padding, segment_article, _init_file_worker, file_segment_worker
Size: 7,739 bytes


### Step 1b.3 — Parallel segmentation (file-based shared-nothing)

**Problem**: applying `segment_article` to 3.2M articles naively via `Pool.imap` requires
pickling ~11 GB of text to workers and ~11 GB of results back through OS pipes.
The pickle serialization/deserialization is single-threaded in the main process and
dominates wall-clock time.

**Solution**: file-based shared-nothing parallelism.
1. **Partition** the clean articles into 100 TSV fragments on disk
2. **Workers** receive only a fragment index (integer) via IPC — they read/write files independently
3. **Main process** concatenates output fragments and assigns sequential IDs

**IPC overhead**: ~400 bytes total (100 integers in + 100 integers out) instead of ~22 GB.

**Resumability**: if the process crashes, workers skip fragments whose output file already exists.

**Directory layout**:
```
data/wikipedia_2018_clean/ordered_fragments/       ← input (100 × ~32K articles)
data/wikipedia_2018_sentence_aligned/ordered_fragments/  ← output (100 × ~210K passages)
```

In [12]:
import polars as pl
from pathlib import Path
import time

# --- Resume guard: ensure output_df and PROJECT_ROOT are available ---
if "output_df" not in dir():
    PROJECT_ROOT = Path.cwd()
    CLEAN_PATH = PROJECT_ROOT / "data" / "wikipedia_2018_clean" / "articles_clean.tsv"
    print(f"Loading articles from {CLEAN_PATH}...")
    output_df = pl.read_csv(CLEAN_PATH, separator="\t")
    print(f"Loaded {len(output_df):,} articles")

if "PROJECT_ROOT" not in dir():
    PROJECT_ROOT = Path.cwd()

# --- Directory setup ---
INPUT_FRAG_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_clean" / "ordered_fragments"
OUTPUT_FRAG_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_sentence_aligned" / "ordered_fragments"

INPUT_FRAG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FRAG_DIR.mkdir(parents=True, exist_ok=True)

# --- Partition articles into 100 fragments ---
# Each fragment is a TSV with columns (title, text) — only what the worker needs.
# The last fragment takes the remainder to avoid losing articles to integer division.

N_FRAGS = 100
n_articles = len(output_df)
chunk_size = n_articles // N_FRAGS

print(f"Partitioning {n_articles:,} articles into {N_FRAGS} fragments")
print(f"  ~{chunk_size:,} articles per fragment")

# Skip if fragments already exist (resumability for the partitioning step too)
existing = list(INPUT_FRAG_DIR.glob("frag_*.tsv"))
if len(existing) == N_FRAGS:
    print(f"\n  All {N_FRAGS} input fragments already exist — skipping partitioning")
else:
    start = time.perf_counter()
    total_written = 0

    for i in range(N_FRAGS):
        offset = i * chunk_size
        # Last fragment takes all remaining articles
        length = chunk_size if i < N_FRAGS - 1 else n_articles - offset

        chunk = output_df.slice(offset, length).select("title", "text")
        chunk.write_csv(INPUT_FRAG_DIR / f"frag_{i}.tsv", separator="\t")
        total_written += len(chunk)

    elapsed = time.perf_counter() - start
    print(f"\n  Written {total_written:,} articles in {elapsed:.1f}s")
    assert total_written == n_articles, f"Mismatch: {total_written} != {n_articles}"
    print(f"  Verification: {total_written:,} == {n_articles:,} ✓")
    print(f"  Output: {INPUT_FRAG_DIR}")

Partitioning 3,232,908 articles into 100 fragments
  ~32,329 articles per fragment

  All 100 input fragments already exist — skipping partitioning


In [13]:
import sys
import time
import importlib
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# === File-based parallel segmentation ===
#
# Why file-based instead of Pool.imap with text data?
#   Pool.imap pickles every article text to workers and pickles results back.
#   For 3.2M articles (~11 GB text + ~11 GB results), the main process spends
#   most of its time in pickle.dumps/loads — single-threaded, on one core.
#
#   File-based: workers read/write files independently. The main process
#   sends only integer fragment indices (100 integers = ~400 bytes total IPC).
#   The actual text data flows through the filesystem, bypassing Python's
#   pickle serialization entirely.
#
# How it works:
#   1. Previous cell partitioned articles into 100 TSV files (input fragments)
#   2. Pool.imap distributes fragment indices [0..99] across workers
#   3. Each worker: reads its input TSV → segment_article per row → writes output TSV
#   4. Worker returns only the passage count (one integer back through the pipe)
#
# Resumability:
#   If the process crashes mid-way, workers skip fragments whose output file
#   already exists. Just re-run this cell to continue from where it stopped.
#
# Load balancing:
#   imap with 100 tasks on 12 workers: workers that finish early pick up the
#   next fragment. Better than static N-way split where one unlucky worker
#   gets all the long articles.

# Ensure project root is importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reload to pick up latest changes to the module
import utils.text_processing
importlib.reload(utils.text_processing)
from utils.text_processing import _init_file_worker, file_segment_worker

n_cores = max(1, cpu_count())

print(f"Segmenting {N_FRAGS} fragments on {n_cores} workers")
print(f"  Input:  {INPUT_FRAG_DIR}")
print(f"  Output: {OUTPUT_FRAG_DIR}")

start = time.perf_counter()

# Pool initializer sets I/O paths in each worker's global state (once per process).
# The actual worker function receives only an integer — minimal IPC.
with Pool(
    n_cores,
    initializer=_init_file_worker,
    initargs=(str(INPUT_FRAG_DIR), str(OUTPUT_FRAG_DIR)),
) as pool:
    results = list(tqdm(
        pool.imap(file_segment_worker, range(N_FRAGS)),
        total=N_FRAGS,
        desc="Segmenting",
        unit="frag",
    ))

elapsed = time.perf_counter() - start

# --- Summary ---
skipped = sum(1 for r in results if r == -1)
processed = sum(1 for r in results if r > 0)
total_passages = sum(r for r in results if r > 0)

print(f"\nDone in {elapsed:.1f}s")
print(f"  Fragments processed: {processed}")
print(f"  Fragments skipped (already existed): {skipped}")
print(f"  Total passages produced: {total_passages:,}")

Segmenting 100 fragments on 24 workers
  Input:  C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_clean\ordered_fragments
  Output: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_sentence_aligned\ordered_fragments


Segmenting: 100%|██████████| 100/100 [00:21<00:00,  4.59frag/s]



Done in 22.4s
  Fragments processed: 100
  Fragments skipped (already existed): 0
  Total passages produced: 23,910,209


In [14]:
import polars as pl
import time

# === Concatenate output fragments → final sentence-aligned corpus ===
#
# Each output fragment is a TSV with columns (text, title) — flat passages.
# We read all 100 in order, concatenate, assign sequential IDs (1-based to
# match DPR convention), and save as a single TSV.

SENTENCE_ALIGNED_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_sentence_aligned"
SENTENCE_ALIGNED_PATH = SENTENCE_ALIGNED_DIR / "psgs_w100_sentence.tsv"

print("Concatenating output fragments...")
start = time.perf_counter()

# Read all fragments in order — guarantees passage ordering matches article ordering
frames = []
for i in range(N_FRAGS):
    frag_path = OUTPUT_FRAG_DIR / f"frag_{i}.tsv"
    assert frag_path.exists(), f"Missing output fragment: {frag_path}"
    frames.append(pl.read_csv(frag_path, separator="\t"))

final_df = pl.concat(frames)

# Free the list of individual DataFrames
del frames

# Add sequential IDs (1-based, matching DPR corpus convention)
# with_row_index adds a UInt32 column; offset=1 makes it 1-based
final_df = final_df.with_row_index("id", offset=1)

# Reorder columns to match DPR format: id, text, title
final_df = final_df.select("id", "text", "title")

# Save
final_df.write_csv(SENTENCE_ALIGNED_PATH, separator="\t")
elapsed = time.perf_counter() - start

size_gb = SENTENCE_ALIGNED_PATH.stat().st_size / (1024 ** 3)
print(f"\nSaved to: {SENTENCE_ALIGNED_PATH}")
print(f"Size: {size_gb:.1f} GB")
print(f"Rows: {len(final_df):,}")
print(f"Time: {elapsed:.1f}s")
print(f"\nSample:")
final_df.head(5)

Concatenating output fragments...

Saved to: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_sentence_aligned\psgs_w100_sentence.tsv
Size: 14.5 GB
Rows: 23,910,209
Time: 27.2s

Sample:


id,text,title
u32,str,str
1,"""Rural Municipality of Chester …","""Rural Municipality of Chester …"
2,"""There is one designated herita…","""Rural Municipality of Chester …"
3,"""As well, the municipality prov…","""Rural Municipality of Chester …"
4,"""Bell Bajao Bell Bajao (Hindi f…","""Bell Bajao"""
5,"""The campaign was launched in I…","""Bell Bajao"""


In [15]:
# === Verification ===

print("=" * 60)
print("VERIFICATION CHECKS")
print("=" * 60)

# 1. Total passage count — should be ~21M (similar to DPR original)
n_passages = len(final_df)
print(f"\n1. Total passages: {n_passages:,}")
print(f"   DPR original had ~21M passages for comparison")

# 2. Word count check — every passage should have exactly 100 words
#    Exception: articles shorter than 100 words (unpadded single segment)
sample = final_df.head(10000)
word_counts = sample["text"].str.count_matches(r" ") + 1
n_not_100 = word_counts.filter(word_counts != 100).len()
print(f"\n2. Word count check (first 10K passages):")
print(f"   Passages with exactly 100 words: {10000 - n_not_100}/10,000")
if n_not_100 > 0:
    print(f"   Passages with != 100 words: {n_not_100} (expected: only very short articles)")

# 3. PAEEK spot-check — known article with 343 clean words → 4 passages
paeek_passages = final_df.filter(pl.col("title") == "PAEEK")
n_paeek = len(paeek_passages)
print(f"\n3. PAEEK spot-check:")
print(f"   Passages found: {n_paeek} (expected: 4)")
assert n_paeek == 4, f"PAEEK mismatch! Got {n_paeek}"
print(f"   ✓ Check passed")

for i, row in enumerate(paeek_passages.iter_rows(named=True)):
    wc = row["text"].count(" ") + 1
    words = row["text"].split(" ")
    print(f"   [{i}] {wc} words | start: {' '.join(words[:6])}...")

# 4. No articles lost — count distinct titles
n_titles = final_df["title"].n_unique()
print(f"\n4. Distinct article titles: {n_titles:,}")
print(f"   Original articles: {n_articles:,}")
# Some titles may have been lost if they had empty text, or gained if not
# This is informational, not a hard assert

# 5. Schema check
print(f"\n5. Schema: {dict(final_df.schema)}")
print(f"   Columns: {final_df.columns}")
assert final_df.columns == ["id", "text", "title"], "Column mismatch!"
print(f"   ✓ Matches DPR format (id, text, title)")

print(f"\n{'=' * 60}")
print("ALL CHECKS PASSED")
print(f"{'=' * 60}")

VERIFICATION CHECKS

1. Total passages: 23,910,209
   DPR original had ~21M passages for comparison

2. Word count check (first 10K passages):
   Passages with exactly 100 words: 10000/10,000

3. PAEEK spot-check:
   Passages found: 4 (expected: 4)
   ✓ Check passed
   [0] 100 words | start: PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi...
   [1] 100 words | start: The PAEEK was a founding member...
   [2] 100 words | start: They were knocked out by PAOK...
   [3] 100 words | start: In the 2011–2012 season they came...

4. Distinct article titles: 3,232,908
   Original articles: 3,232,908

5. Schema: {'id': UInt32, 'text': String, 'title': String}
   Columns: ['id', 'text', 'title']
   ✓ Matches DPR format (id, text, title)

ALL CHECKS PASSED
